In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Step 1: Setup Environment and Import Libraries

In [ ]:
pip install pandas numpy scikit-learn tensorflow keras opencv-python

# **Now, import the necessary libraries in your Python script or Jupyter notebook:**

In [ ]:
import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

# Your TensorFlow code here


In [ ]:
# Load labels
labels_df = pd.read_csv(label_file)

# Print column names to verify
print(labels_df.columns)


In [ ]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Define the paths to the images and labels
image_folder = '/kaggle/input/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Training/training_words'
label_file = '/kaggle/input/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Training/training_labels.csv'

# Load labels
labels_df = pd.read_csv(label_file)

# Print column names to verify
print("Column names in the CSV file:", labels_df.columns)

# Initialize lists for images and labels
images = []
labels = []

# Correct the column names based on the printed output
filename_col = 'IMAGE'  # Replace with actual column name if different
label_col = 'MEDICINE_NAME'        # Replace with actual column name if different

# Debug: print the first few rows to understand the structure
print("First few rows of the dataframe:")
print(labels_df.head())

# Check if the specified columns exist
if filename_col not in labels_df.columns or label_col not in labels_df.columns:
    raise ValueError(f"Specified columns '{filename_col}' or '{label_col}' not found in the CSV file.")

# Load images and their corresponding labels
for index, row in labels_df.iterrows():
    img_path = os.path.join(image_folder, row[filename_col])
    img = load_img(img_path, target_size=(64, 64))  # Resize to a fixed size
    img = img_to_array(img)
    images.append(img)
    labels.append(row[label_col])

# Convert lists to numpy arrays
images = np.array(images, dtype="float") / 255.0  # Normalize pixel values
labels = np.array(labels)

# Encode labels
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded, num_classes=78)  # 78 classes

# Split the data
X_train, X_test, y_train, y_test = train_test_split(images, labels_categorical, test_size=0.2, random_state=42)

# Build the model
model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(78, activation='softmax'))  # 78 classes

# Compile the model
model.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

# Save the model
model.save('prescription_classification_model.h5')


In [ ]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Define the paths to the images and labels
image_folder = '/kaggle/input/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Training/training_words'
label_file = '/kaggle/input/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Training/training_labels.csv'

# Load labels
labels_df = pd.read_csv(label_file)

# Column names in the CSV file
filename_col = 'IMAGE'
label_col = 'MEDICINE_NAME'

# Debug: print the first few rows to understand the structure
print("First few rows of the dataframe:")
print(labels_df.head())

# Initialize lists for images and labels
images = []
labels = []

# Load images and their corresponding labels
for index, row in labels_df.iterrows():
    img_path = os.path.join(image_folder, row[filename_col])
    img = load_img(img_path, target_size=(64, 64))  # Resize to a fixed size
    img = img_to_array(img)
    images.append(img)
    labels.append(row[label_col])

# Convert lists to numpy arrays
images = np.array(images, dtype="float") / 255.0  # Normalize pixel values
labels = np.array(labels)

# Encode labels
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(labels)
labels_categorical = to_categorical(labels_encoded, num_classes=78)  # 78 classes

# Split the data
X_train, X_test, y_train, y_test = train_test_split(images, labels_categorical, test_size=0.2, random_state=42)

# Build the model
model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(78, activation='softmax'))  # 78 classes

# Compile the model
model.compile(optimizer=Adam(lr=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=20, batch_size=32)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

# Save the model in the newer format
model.save('prescription_classification_model.keras')


In [ ]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model

# Define the paths to the test images and labels
test_image_folder = '/kaggle/input/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Testing/testing_words'
test_label_file = '/kaggle/input/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Testing/testing_labels.csv'

# Load test labels
test_labels_df = pd.read_csv(test_label_file)

# Column names in the CSV file
filename_col = 'IMAGE'
label_col = 'MEDICINE_NAME'

# Debug: print the first few rows to understand the structure
print("First few rows of the test dataframe:")
print(test_labels_df.head())

# Initialize lists for test images and labels
test_images = []
test_labels = []

# Load test images and their corresponding labels
for index, row in test_labels_df.iterrows():
    img_path = os.path.join(test_image_folder, row[filename_col])
    img = load_img(img_path, target_size=(64, 64))  # Resize to a fixed size
    img = img_to_array(img)
    test_images.append(img)
    test_labels.append(row[label_col])

# Convert lists to numpy arrays
test_images = np.array(test_images, dtype="float") / 255.0  # Normalize pixel values
test_labels = np.array(test_labels)

# Encode test labels
label_encoder = LabelEncoder()
test_labels_encoded = label_encoder.fit_transform(test_labels)
test_labels_categorical = tf.keras.utils.to_categorical(test_labels_encoded, num_classes=78)  # 78 classes

# Load the trained model
model = load_model('prescription_classification_model.keras')

# Evaluate the model on the test data
loss, accuracy = model.evaluate(test_images, test_labels_categorical)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

# Optionally: Predict and show a confusion matrix
from sklearn.metrics import classification_report, confusion_matrix

# Predict the classes
predictions = model.predict(test_images)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(test_labels_categorical, axis=1)

# Generate classification report
print(classification_report(true_classes, predicted_classes, target_names=label_encoder.classes_))

# Generate confusion matrix
conf_matrix = confusion_matrix(true_classes, predicted_classes)
print("Confusion Matrix:")
print(conf_matrix)


In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model

def load_and_prepare_test_data(test_image_folder, test_label_file, target_size=(64, 64)):
    # Load test labels
    test_labels_df = pd.read_csv(test_label_file)

    # Column names in the CSV file
    filename_col = 'IMAGE'
    label_col = 'MEDICINE_NAME'

    # Initialize lists for test images and labels
    test_images = []
    test_labels = []

    # Load test images and their corresponding labels
    for index, row in test_labels_df.iterrows():
        img_path = os.path.join(test_image_folder, row[filename_col])
        img = load_img(img_path, target_size=target_size)  # Resize to a fixed size
        img = img_to_array(img)
        test_images.append(img)
        test_labels.append(row[label_col])

    # Convert lists to numpy arrays
    test_images = np.array(test_images, dtype="float") / 255.0  # Normalize pixel values
    test_labels = np.array(test_labels)

    # Encode test labels
    label_encoder = LabelEncoder()
    test_labels_encoded = label_encoder.fit_transform(test_labels)
    test_labels_categorical = tf.keras.utils.to_categorical(test_labels_encoded, num_classes=len(label_encoder.classes_))

    return test_images, test_labels_categorical, label_encoder

def evaluate_model(model_path, test_images, test_labels_categorical, label_encoder):
    # Load the trained model
    model = load_model(model_path)

    # Evaluate the model on the test data
    loss, accuracy = model.evaluate(test_images, test_labels_categorical)
    print(f'Test Accuracy: {accuracy * 100:.2f}%')

    # Predict the classes
    predictions = model.predict(test_images)
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(test_labels_categorical, axis=1)

    # Generate classification report
    from sklearn.metrics import classification_report, confusion_matrix
    print(classification_report(true_classes, predicted_classes, target_names=label_encoder.classes_))

    # Generate confusion matrix
    conf_matrix = confusion_matrix(true_classes, predicted_classes)
    print("Confusion Matrix:")
    print(conf_matrix)

# Paths to your test data
test_image_folder = '/kaggle/input/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Testing/testing_words'
test_label_file = '/kaggle/input/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset/Testing/testing_labels.csv'
model_path = 'prescription_classification_model.keras'

# Load and prepare test data
test_images, test_labels_categorical, label_encoder = load_and_prepare_test_data(test_image_folder, test_label_file)

# Evaluate the model on the new test data
evaluate_model(model_path, test_images, test_labels_categorical, label_encoder)


In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')


In [ ]:
# Save the model
model.save('prescription_classification_model.h5')


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# Fit the generator to the training data
datagen.fit(X_train)

# Train the model with augmented data
history = model.fit(datagen.flow(X_train, y_train, batch_size=32),
                    validation_data=(X_test, y_test), epochs=20)
